<a href="https://colab.research.google.com/github/lydiacyhung/114-2-ProgramingLanguage/blob/main/HW4_PTT_GoogleSheet_RAG%E6%95%B4%E7%90%86%E7%89%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW4：PTT → Google Sheet → RAG（整理版）

這份 notebook 保留完整流程：

1. 爬取 PTT movie 文章
2. 寫入指定 Google Sheet
3. 從 Google Sheet 讀回資料
4. 建立 FAISS RAG 索引
5. 用 Gemini 根據 PTT 資料回答問題

主要修正：原本設定了 `SHEET_URL`，但實際用 `gc.open(WORKSHEET_NAME)` 開啟試算表，容易打開錯的 Spreadsheet。新版固定使用 `gc.open_by_url(SHEET_URL)`。


In [1]:
# 安裝必要套件
!pip -q install gspread gspread-dataframe faiss-cpu sentence-transformers beautifulsoup4 requests google-generativeai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 55.3 MB/s eta 0:00:00


In [ ]:
import re
import time
import uuid
from datetime import datetime
from urllib.parse import urljoin

import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

import faiss
from sentence_transformers import SentenceTransformer

import google.generativeai as genai
from google.colab import auth, userdata
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe, get_as_dataframe
import gradio as gr

## 1. 基本設定

請確認 `SHEET_URL` 是你要寫入的 Google Sheet。  
`PTT_WORKSHEET_NAME` 是存放 PTT 原始文章的分頁。


In [3]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1fc5laOITajG78TNIXouEP2ernmL9MKvi5VJb2jVWQyw/edit?gid=440923892#gid=440923892"
PTT_WORKSHEET_NAME = "HW4"
TIMEZONE_NOTE = "Asia/Taipei"

PTT_HEADER = [
    "post_id", "title", "url", "date", "author", "nrec",
    "created_at", "fetched_at", "content"
]

PTT_MOVIE_INDEX = "https://www.ptt.cc/bbs/movie/index.html"
PTT_COOKIES = {"over18": "1"}
USER_AGENT = "Mozilla/5.0 (compatible; Colab PTT crawler)"


## 2. 連線 Google Sheet

這裡是最重要的修正：使用 `open_by_url(SHEET_URL)`，不要用 worksheet 名稱打開 spreadsheet。


In [4]:
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 關鍵修正：直接用網址開啟指定 Google Sheet
sh = gc.open_by_url(SHEET_URL)
print(f"✅ 已開啟試算表：{sh.title}")
print(f"🔗 {SHEET_URL}")


✅ 已開啟試算表：程式設計資料庫
🔗 https://docs.google.com/spreadsheets/d/1fc5laOITajG78TNIXouEP2ernmL9MKvi5VJb2jVWQyw/edit?gid=440923892#gid=440923892


In [5]:
def ensure_worksheet(spreadsheet, title, header, rows=1000):
    """取得或建立 worksheet，並確保表頭正確。"""
    try:
        ws = spreadsheet.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = spreadsheet.add_worksheet(title=title, rows=str(rows), cols=str(len(header) + 5))
        ws.update([header])
        return ws

    values = ws.get_all_values()
    if not values:
        ws.update([header])
    elif values[0] != header:
        # 保留資料但重建欄位較危險，因此這裡直接清掉並重新建立正確表頭。
        # 若你要保留舊資料，請先備份 Google Sheet。
        ws.clear()
        ws.update([header])
    return ws


def read_sheet_df(ws, header):
    """從 worksheet 讀成 DataFrame，並清掉空列。"""
    df = get_as_dataframe(ws, evaluate_formulas=True, dtype=str).dropna(how="all")
    if df.empty:
        return pd.DataFrame(columns=header)
    df = df.loc[:, [c for c in df.columns if not str(c).startswith("Unnamed")]]
    for col in header:
        if col not in df.columns:
            df[col] = ""
    return df[header].fillna("")


def write_sheet_df(ws, df, header):
    """把 DataFrame 寫回 worksheet。"""
    df_out = df.copy()
    for col in header:
        if col not in df_out.columns:
            df_out[col] = ""
    df_out = df_out[header].infer_objects(copy=False).fillna("") # Add infer_objects to address FutureWarning

    # Google Sheet 寫入前統一轉字串，避免 Timestamp / NaN 型別問題
    for c in df_out.columns:
        df_out[c] = df_out[c].astype(str)

    ws.clear()
    set_with_dataframe(ws, df_out, include_index=False, include_column_header=True, resize=True)
    return len(df_out)


ws_ptt = ensure_worksheet(sh, PTT_WORKSHEET_NAME, PTT_HEADER)
print(f"✅ 已準備 worksheet：{ws_ptt.title}")

✅ 已準備 worksheet：HW4


## 3. PTT movie 爬蟲

這段只負責爬 PTT，不碰 RAG。資料會先存在 `new_posts_df`。


In [6]:
def now_iso():
    return datetime.now().isoformat(timespec="seconds")


def get_soup(url):
    resp = requests.get(
        url,
        timeout=20,
        headers={"User-Agent": USER_AGENT},
        cookies=PTT_COOKIES,
    )
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def get_prev_index_url(soup):
    for a in soup.select("div.btn-group-paging a.btn.wide"):
        if "上頁" in a.get_text(strip=True):
            href = a.get("href")
            return urljoin("https://www.ptt.cc", href) if href else None
    return None


def parse_nrec(nrec_span):
    if not nrec_span:
        return 0
    txt = nrec_span.get_text(strip=True)
    if txt == "爆":
        return 100
    if txt.startswith("X"):
        try:
            return -int(txt[1:])
        except Exception:
            return -10
    try:
        return int(txt)
    except Exception:
        return 0


def extract_post_list(index_soup):
    posts = []
    for item in index_soup.select("div.r-ent"):
        a = item.select_one("div.title a")
        if not a:
            continue

        title = a.get_text(strip=True)
        url = urljoin("https://www.ptt.cc", a.get("href"))
        author_node = item.select_one("div.author")
        date_node = item.select_one("div.date")
        nrec_node = item.select_one("div.nrec span")

        posts.append({
            "title": title,
            "url": url,
            "author": author_node.get_text(strip=True) if author_node else "",
            "date": date_node.get_text(strip=True) if date_node else "",
            "nrec": parse_nrec(nrec_node),
        })
    return posts


def clean_ptt_content(article_soup):
    main = article_soup.select_one("#main-content")
    if not main:
        return "", ""

    # 取出文章建立時間
    created_at = ""
    metalines = main.select("div.article-metaline")
    for m in metalines:
        tag = m.select_one("span.article-meta-tag")
        value = m.select_one("span.article-meta-value")
        if tag and value and tag.get_text(strip=True) == "時間":
            created_at = value.get_text(strip=True)

    # 移除 meta 與推文
    for node in main.select("div.article-metaline, div.article-metaline-right, div.push"):
        node.decompose()

    text = main.get_text("\n", strip=True)
    text = re.split(r"\n--\n|\n※ 發信站:", text)[0].strip()
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text, created_at


def make_post_id(url):
    # 用文章網址檔名當 post_id，穩定且方便去重
    return url.rstrip("/").split("/")[-1].replace(".html", "")


def crawl_ptt_movie(pages=2, delay=0.5):
    """爬取 PTT movie 最新 pages 頁文章。"""
    all_rows = []
    index_url = PTT_MOVIE_INDEX

    for page in range(int(pages)):
        print(f"📄 正在讀取列表頁 {page + 1}/{pages}: {index_url}")
        index_soup = get_soup(index_url)
        post_list = extract_post_list(index_soup)

        for p in post_list:
            try:
                article_soup = get_soup(p["url"])
                content, created_at = clean_ptt_content(article_soup)
                row = {
                    "post_id": make_post_id(p["url"]),
                    "title": p["title"],
                    "url": p["url"],
                    "date": p["date"],
                    "author": p["author"],
                    "nrec": p["nrec"],
                    "created_at": created_at,
                    "fetched_at": now_iso(),
                    "content": content,
                }
                all_rows.append(row)
                time.sleep(delay)
            except Exception as e:
                print(f"⚠️ 跳過文章：{p.get('title', '')}，原因：{e}")

        prev_url = get_prev_index_url(index_soup)
        if not prev_url:
            break
        index_url = prev_url
        time.sleep(delay)

    df = pd.DataFrame(all_rows, columns=PTT_HEADER)
    print(f"✅ 本次爬到 {len(df)} 篇文章")
    return df

## 4. 執行爬蟲並寫入 Google Sheet

這一格會：

1. 從 Google Sheet 讀取既有資料
2. 爬取新的 PTT 資料
3. 合併並用 `post_id` 去重
4. 寫回 Google Sheet
5. 再讀一次確認真的寫入成功


In [8]:
# 你可以調整 pages，例如 pages=1 先測試，確認成功後再改成 3 或 5
new_posts_df = crawl_ptt_movie(pages=2, delay=2.0) # Increased delay to reduce connection issues

old_posts_df = read_sheet_df(ws_ptt, PTT_HEADER)
print(f"📌 Google Sheet 原本有 {len(old_posts_df)} 筆")

ptt_posts_df = pd.concat([old_posts_df, new_posts_df], ignore_index=True)
ptt_posts_df = ptt_posts_df.drop_duplicates(subset=["post_id"], keep="last")
ptt_posts_df = ptt_posts_df.sort_values(by="fetched_at", ascending=False)

written_count = write_sheet_df(ws_ptt, ptt_posts_df, PTT_HEADER)
print(f"✅ 已寫入 Google Sheet：{written_count} 筆")

verify_df = read_sheet_df(ws_ptt, PTT_HEADER)
print(f"🔍 從 Google Sheet 重新讀回：{len(verify_df)} 筆")

if len(verify_df) == written_count:
    print("✅ 寫入驗證成功")
else:
    print("⚠️ 寫入筆數與讀回筆數不同，請檢查 Google Sheet 權限或資料格式")

📄 正在讀取列表頁 1/2: https://www.ptt.cc/bbs/movie/index.html
📄 正在讀取列表頁 2/2: https://www.ptt.cc/bbs/movie/index11002.html
✅ 本次爬到 35 篇文章
📌 Google Sheet 原本有 0 筆
✅ 已寫入 Google Sheet：35 筆
🔍 從 Google Sheet 重新讀回：35 筆
✅ 寫入驗證成功


## 5. 從 Google Sheet 建立 RAG 索引

重點：RAG 不直接吃剛爬下來的記憶體資料，而是**從 Google Sheet 重新讀回**，這樣才能確認流程真的是：

`PTT → Google Sheet → RAG`


In [9]:
# 從 Google Sheet 重新讀取，作為 RAG 的唯一資料來源
rag_source_df = read_sheet_df(ws_ptt, PTT_HEADER)

# 清掉沒有內容的文章
rag_source_df = rag_source_df[rag_source_df["content"].astype(str).str.strip() != ""].copy()
print(f"📚 可用於 RAG 的文章數：{len(rag_source_df)}")

rag_source_df.head()


📚 可用於 RAG 的文章數：35


,post_id,title,url,date,author,nrec,created_at,fetched_at,content
0,M.1780415687.A.4C2,[新聞] <妳最後留下的歌>道枝駿佑演繹少年普通感,https://www.ptt.cc/bbs/movie/M.1780415687.A.4C...,06/02,kkaicd1,0,Tue Jun 2 23:54:45 2026,2026-06-04 1:29:41,新聞網址：\nhttps://www.ccii.com.tw/html/news_conte...
1,M.1780415337.A.089,[新聞] 話題恐怖片《後室》吸引Z世代進入電影院 Youtube社群現象將,https://www.ptt.cc/bbs/movie/M.1780415337.A.08...,06/02,hihihihehehe,0,Tue Jun 2 23:48:54 2026,2026-06-04 1:29:38,話題恐怖片《後室》吸引Z世代進入電影院 Youtube社群現象將重塑好萊塢生態\n\n19:...
2,M.1780415116.A.54B,[新聞]元華曝成龍真實性格！與洪金寶對比鮮明！,https://www.ptt.cc/bbs/movie/M.1780415116.A.54...,06/02,XDGEE,55,Tue Jun 2 23:45:14 2026,2026-06-04 1:29:35,元華曝成龍真實性格！與洪金寶對比鮮明！\n香港資深武打演員元華（原名容繼志），與同為「七小福...
3,M.1780415074.A.C81,[新聞]《後室》票房大賣1.18億美元！但續集電影,https://www.ptt.cc/bbs/movie/M.1780415074.A.C8...,06/02,XDGEE,6,Tue Jun 2 23:44:32 2026,2026-06-04 1:29:33,《後室》票房大賣1.18億美元！但續集電影可能不出！導演想改拍電視劇，故事改聚焦「科\n技驚...
4,M.1780414713.A.AA1,[好雷]一個妥瑞氏症患者看《出口成髒》,https://www.ptt.cc/bbs/movie/M.1780414713.A.AA...,06/02,soltea,74,Tue Jun 2 23:38:31 2026,2026-06-04 1:29:30,我是5/23看口碑場的，本來沒有打算寫文，但影評看一看，好像很少看到有妥瑞氏症患者\n看這部...


In [10]:
print("正在載入多語言 Embedding 模型...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("✅ Embedding 模型載入完成")


def build_rag_documents(df):
    docs = []
    for _, row in df.iterrows():
        title = str(row.get("title", ""))
        content = str(row.get("content", ""))
        url = str(row.get("url", ""))
        author = str(row.get("author", ""))
        date = str(row.get("date", ""))
        nrec = str(row.get("nrec", ""))

        text = (f"標題：{title}\n"
                f"作者：{author}\n"
                f"日期：{date}\n"
                f"推文數：{nrec}\n"
                f"內容：{content}")
        docs.append({
            "post_id": str(row.get("post_id", "")),
            "title": title,
            "url": url,
            "text": text,
        })
    return docs


def build_faiss_index(docs):
    if not docs:
        raise ValueError("沒有可建立索引的文件")

    texts = [d["text"] for d in docs]
    embeddings = embedding_model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
    embeddings = embeddings.astype("float32")

    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index, embeddings


rag_documents = build_rag_documents(rag_source_df)
rag_index, rag_embeddings = build_faiss_index(rag_documents)

print(f"✅ RAG 索引建立完成：{len(rag_documents)} 篇文章，向量維度 {rag_embeddings.shape[1]}")

正在載入多語言 Embedding 模型...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding 模型載入完成


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

✅ RAG 索引建立完成：35 篇文章，向量維度 384


## 6. Gemini 設定與 RAG 問答

請先在 Colab Secrets 裡建立 `gemini`，內容是你的 Gemini API key。


In [13]:
api_key = userdata.get("Gemini")
if not api_key:
    raise ValueError("找不到 Colab Secret：gemini。請先在 Colab Secrets 新增 Gemini API key。")

genai.configure(api_key=api_key)

# 若你的帳號不支援這個模型，可改成你可用的 Gemini model name
GEMINI_MODEL_NAME = "gemini-3-flash-preview"
llm = genai.GenerativeModel(GEMINI_MODEL_NAME)
print(f"✅ Gemini 已設定：{GEMINI_MODEL_NAME}")


✅ Gemini 已設定：gemini-3-flash-preview


In [14]:
def retrieve_docs(query, k=3):
    if rag_index is None or not rag_documents:
        return []
    q_emb = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = rag_index.search(q_emb, int(k))

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1:
            continue
        doc = rag_documents[idx].copy()
        doc["distance"] = float(dist)
        results.append(doc)
    return results


def query_rag(question, k=3):
    docs = retrieve_docs(question, k=k)
    if not docs:
        return "找不到相關 PTT 資料。"

    context = "\n\n---\n\n".join(
        [f"來源標題：{d['title']}\n來源網址：{d['url']}\n{d['text']}" for d in docs]
    )

    prompt = f"""
你是一個根據 PTT 電影版資料回答問題的助教。
請只根據【PTT 資料】回答。
如果資料不足，請明確說「目前資料不足，無法判斷」，不要自行編造。
回答請使用繁體中文，並在最後列出參考來源標題與網址。

【PTT 資料】
{context}

【問題】
{question}

【回答】
""".strip()

    response = llm.generate_content(prompt)
    return response.text

## 7.1 Gradio 輔助函式定義

由於 Gradio 介面依賴多個輔助函式來處理任務、番茄鐘、爬蟲和 RAG 相關邏輯，這些函式需要在介面定義之前被載入。這裡提供了這些函式的基本定義，以避免 `NameError`。

In [31]:
# 初始化全域 DataFrame 變數，避免 NameError
tasks_df = pd.DataFrame(columns=["task", "priority", "est_min", "due_date", "labels", "notes", "planned_for", "status"])
logs_df = pd.DataFrame(columns=["timestamp", "type", "task", "cycles", "duration", "note"])
clips_df = pd.DataFrame(columns=["content", "source_url"])


In [32]:
# 替 Gradio 介面所需的函式提供基本定義，若有完整邏輯請替換

def refresh_all():
    # 這裡應該是從 Google Sheet 讀取最新資料的邏輯
    # 由於沒有完整的 Google Sheet 互動邏輯，這裡回傳空的 DataFrame
    return (
        pd.DataFrame(columns=["task", "priority", "est_min", "due_date", "labels", "notes", "planned_for", "status"]),
        pd.DataFrame(columns=["timestamp", "type", "task", "cycles", "duration", "note"]),
        pd.DataFrame(columns=["content", "source_url"])
    )

def today_summary():
    # 提供一個簡單的總結文字
    return "今日任務總結：尚無資料。"

def list_task_choices():
    # 提供預設的任務選項
    return ["任務 A", "任務 B", "任務 C"]

def add_task(task, priority, est_min, due_date, labels, notes, planned_for):
    print(f"新增任務：{task}")
    # 這裡應該有將任務寫入 Google Sheet 的邏輯
    return f"新增任務 '{task}' 成功！", tasks_df

def update_task_status(task_choice, new_status):
    print(f"更新任務 '{task_choice}' 狀態為 '{new_status}'")
    # 這裡應該有更新 Google Sheet 中任務狀態的邏輯
    return f"任務 '{task_choice}' 狀態已更新為 '{new_status}'", tasks_df

def mark_done(task_choice):
    print(f"標記任務 '{task_choice}' 為完成")
    # 這裡應該有更新 Google Sheet 中任務狀態的邏輯
    return f"任務 '{task_choice}' 已標記為完成", tasks_df

def start_phase(sel_task, phase_type, cycles):
    print(f"開始 {phase_type} 階段，任務：{sel_task}，番茄數：{cycles}")
    # 這裡應該有記錄番茄鐘開始時間的邏輯
    return f"開始 {phase_type} 階段，任務 '{sel_task}'"

def end_phase(sel_task, note):
    print(f"結束階段，任務：{sel_task}，備註：{note}")
    # 這裡應該有記錄番茄鐘結束時間和備註的邏輯
    return f"結束階段，任務 '{sel_task}' 已記錄。"

def generate_today_plan():
    # 這裡應該有使用 AI 產生任務計畫的邏輯
    return "今日計畫：尚無 AI 計畫。"


In [33]:
# 替 RAG 介面所需的函式提供基本定義，若有完整邏輯請替換

def build_faiss_index_from_df(df):
    if df.empty:
        return "⚠️ 無資料可建立索引。"
    try:
        # 假設 rag_documents 和 rag_index 是全域變數，或是可以從外部獲取
        global rag_documents, rag_index, rag_embeddings
        rag_documents = build_rag_documents(df)
        rag_index, rag_embeddings = build_faiss_index(rag_documents)
        return f"✅ 已從 {len(rag_documents)} 篇文章建立 RAG 索引。"
    except Exception as e:
        return f"❌ 建立 RAG 索引失敗：{e}"

def get_confidence_label(avg_distance):
    # 根據距離提供信心度標籤，這是一個簡單的範例
    if avg_distance < 1.0:
        return "高"
    elif avg_distance < 2.0:
        return "中"
    else:
        return "低"

def build_chat_prompt(question, docs, history):
    # 構建給 Gemini 模型的提示詞
    context = "\n\n---\n\n".join(
        [f"來源標題：{d['title']}\n來源網址：{d['url']}\n{d['text']}" for d in docs]
    )
    history_str = "\n".join([f"{h['role']}: {h['content']}" for h in history])

    prompt = f"""
你是一個根據 PTT 電影版資料回答問題的助教。
請只根據【PTT 資料】回答。
如果資料不足，請明確說「目前資料不足，無法判斷」，不要自行編造。
回答請使用繁體中文，並在最後列出參考來源標題與網址。

【對話歷史】
{history_str}

【PTT 資料】
{context}

【問題】
{question}

【回答】
""".strip()
    return prompt

def log_qa(session_id, turn, question, answer, sources, confidence):
    print(f"[Log QA] Session: {session_id}, Turn: {turn}, Q: {question[:30]}..., A: {answer[:30]}...")
    # 這裡應該有將 QA 紀錄寫入 Google Sheet 或資料庫的邏輯

In [ ]:
import gradio as gr

# ===============================
# 1️⃣ 資料刷新
# ===============================
def _refresh():
    global tasks_df, logs_df, clips_df
    tasks_df, logs_df, clips_df = refresh_all()
    return tasks_df.head(50), logs_df.head(50), today_summary()  # 限制顯示列數

# ===============================
# 2️⃣ 包裝 RAG 問答 (進度提示)
# ===============================
def _gradio_query_rag(question, k):
    if not question.strip():
        yield "⚠️ 請輸入問題再開始檢索。"
        return
    yield "🔍 正在檢索 PTT 資料庫並產生回答，請稍候..."
    try:
        # 這裡假設 query_rag 是同步函式，如果很耗時，可以換成非同步或用 ThreadPoolExecutor
        ans = query_rag(question, k)
        yield ans
    except Exception as e:
        yield f"❌ 發生錯誤：{str(e)}"

# ===============================
# 3️⃣ RAG 索引建立 (分離按鈕)
# ===============================
def _build_rag_index(pages):
    yield f"ℹ️ 開始爬取 PTT 最新 {pages} 頁文章..."
    try:
        status, count = crawl_ptt_movie(index_pages=int(pages), min_push=10)
        yield f"✅ 爬取完成，抓到文章數量: {count}"
        yield "⚙️ 正在建立 FAISS 向量索引..."
        msg, index = build_faiss_index_from_ptt_data()
        yield msg
        yield "🎉 RAG 資料庫更新完成！"
    except Exception as e:
        yield f"❌ 發生錯誤：{str(e)}"

# ===============================
# 4️⃣ Gradio 介面
# ===============================
with gr.Blocks(title="待辦清單＋番茄鐘＋PTT 電影 RAG 系統") as demo:
    gr.Markdown("# ✅ 待辦任務管理 與 PTT 電影版 RAG 知識庫系統")

    # --- 刷新 ---
    with gr.Row():
        btn_refresh = gr.Button("🔄 重新整理（Sheet → App）")
        out_summary = gr.Markdown(today_summary())

    # --- Tasks 分頁 ---
    with gr.Tab("Tasks"):
        with gr.Row():
            with gr.Column(scale=2):
                task = gr.Textbox(label="任務名稱", placeholder="寫 HW3 報告 / 修正 SQL / …")
                priority = gr.Dropdown(["H","M","L"], value="M", label="優先級")
                est_min = gr.Number(value=25, label="預估時間（分鐘）", precision=0)
                due_date = gr.Textbox(label="到期日（YYYY-MM-DD，可空白）")
                labels = gr.Textbox(label="標籤（逗號分隔，可空白）")
                notes = gr.Textbox(label="備註（可空白）")
                planned_for = gr.Dropdown(["","today","tomorrow"], value="", label="規劃歸屬")
                btn_add = gr.Button("➕ 新增任務")
                msg_add = gr.Markdown()
            with gr.Column(scale=3):
                grid_tasks = gr.Dataframe(value=tasks_df.head(50), label="任務清單", interactive=False)
        with gr.Row():
            task_choice = gr.Dropdown(choices=list_task_choices(), label="選取任務（用於更新）")
            new_status = gr.Dropdown(["todo","in-progress","done"], value="in-progress", label="更新狀態")
            btn_update = gr.Button("✏️ 更新狀態")
            btn_done = gr.Button("✅ 直接標記完成")
            msg_update = gr.Markdown()

    # --- Pomodoro 分頁 ---
    with gr.Tab("Pomodoro"):
        with gr.Row():
            sel_task = gr.Dropdown(choices=list_task_choices(), label="選擇任務")
            cycles = gr.Number(value=1, precision=0, label="番茄數（僅作紀錄）")
        with gr.Row():
            btn_start_work = gr.Button("▶️ 開始工作")
            note_work = gr.Textbox(label="工作備註（可空白）")
            btn_end_work = gr.Button("⏹️ 結束工作並記錄")
        with gr.Row():
            btn_start_break = gr.Button("🍵 開始休息")
            note_break = gr.Textbox(label="休息備註（可空白）")
            btn_end_break = gr.Button("⏹️ 結束休息並記錄")
        msg_pomo = gr.Markdown()
        grid_logs = gr.Dataframe(value=logs_df.head(50), label="番茄鐘紀錄", interactive=False)

    # --- AI Plan 分頁 ---
    with gr.Tab("AI Plan"):
        gr.Markdown("把**今天的任務**排成 **morning / afternoon / evening** 三段行動計畫。")
        btn_plan = gr.Button("🧠 產生今日計畫")
        out_plan = gr.Markdown()

    # --- PTT 電影 RAG 分頁 ---
    with gr.Tab("🎬 PTT 電影 RAG 問答"):
        gr.Markdown("### 🕷️ 步驟 1：爬取 PTT 文章並更新 RAG 資料庫")
        with gr.Row():
            rag_pages = gr.Number(value=2, label="要爬取 PTT 最新幾頁文章？", precision=0)
            btn_build_rag = gr.Button("🚀 更新 RAG 索引")
        out_rag_status = gr.Markdown("ℹ️ 系統啟動後，請先執行 RAG 索引建立。")

        gr.Markdown("### 💬 步驟 2：電影問題檢索")
        with gr.Row():
            rag_query = gr.Textbox(label="輸入你的電影問題", placeholder="例如：最近推薦的恐怖片", lines=2)
            rag_topk = gr.Slider(minimum=1, maximum=5, value=3, step=1, label="參考文本篇數 (Top K)")
        btn_rag_chat = gr.Button("🧠 AI 回答問題")
        out_rag_answer = gr.Markdown("### 【AI 助理回答】")

    # ===============================
    # 事件綁定
    # ===============================
    btn_refresh.click(_refresh, outputs=[grid_tasks, grid_logs, out_summary])
    btn_add.click(add_task, inputs=[task, priority, est_min, due_date, labels, notes, planned_for], outputs=[msg_add, grid_tasks])
    btn_update.click(update_task_status, inputs=[task_choice, new_status], outputs=[msg_update, grid_tasks])
    btn_done.click(mark_done, inputs=[task_choice], outputs=[msg_update, grid_tasks])
    btn_start_work.click(start_phase, inputs=[sel_task, gr.State('work'), cycles], outputs=[msg_pomo])
    btn_end_work.click(end_phase, inputs=[sel_task, note_work], outputs=[msg_pomo])
    btn_plan.click(generate_today_plan, outputs=[out_plan])

    # RAG 事件綁定
    btn_build_rag.click(_build_rag_index, inputs=[rag_pages], outputs=[out_rag_status])
    btn_rag_chat.click(_gradio_query_rag, inputs=[rag_query, rag_topk], outputs=[out_rag_answer])

# 啟動
demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5c34bffc2fb024ff05.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 7. 快速測試


## 常見錯誤檢查

如果 PTT 資料沒有寫回 Google Sheet，請依序檢查：

1. 是否有成功印出 `已開啟試算表`，且名稱正確。
2. `SHEET_URL` 是否是你要寫入的那一份 Google Sheet。
3. Google Sheet 權限是否允許目前 Colab 登入的 Google 帳號編輯。
4. 是否執行到「執行爬蟲並寫入 Google Sheet」那一格。
5. 是否有看到 `寫入驗證成功`。
6. RAG 要從 `rag_source_df = read_sheet_df(...)` 開始，確保資料來源是 Google Sheet，而不是記憶體中的暫存變數。
